In [1]:
import torch
from torch.utils import data
from torch import nn
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import Subset
from torch.nn import functional as F
from torch.optim.swa_utils import update_bn
import os
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from PIL import Image
from torch.utils import data
from torchvision import transforms
k = 5
lr = 0.05
epoch = 40
batch_size = 64
weight_decay = 5e-4
momentum = 0.9
image_size = 224
seed = 42
loader_config = {
    "batch_size": batch_size,
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": True}




import torch
from torch.utils import data
from torch import nn
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import Subset
from torch.nn import functional as F
from torch.optim.swa_utils import update_bn
import os
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from PIL import Image
from torch.utils import data
from torchvision import transforms
k = 5
lr = 0.05
epoch = 40
batch_size = 64
weight_decay = 5e-4
momentum = 0.9
image_size = 224
seed = 42
loader_config = {
    "batch_size": batch_size,
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": True

}
import random

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [ ]:
import os
import pandas as pd

from PIL import Image
from torch.utils import data
from torchvision import transforms


class LeavesDataset(data.Dataset):
    def __init__(
        self,
        dataframe,
        data_dir,
        transform,
        class_to_idx
    ):
        super().__init__()

        # 重置行号，保证Dataset索引从0开始连续排列
        self.dataframe = dataframe.reset_index(drop=True)

        self.data_dir = data_dir
        self.transform = transform

        # 按照CSV原有顺序保存图片相对路径
        self.image_paths = self.dataframe["image"].tolist()

        # 类别名称和数字之间的映射
        self.class_to_idx = class_to_idx

        self.classes = list(
            class_to_idx.keys()
        )

        self.idx_to_class = {
            idx: label
            for label, idx in class_to_idx.items()
        }

        # 判断当前是训练集还是测试集
        self.has_label = (
            "label" in self.dataframe.columns
        )

        # 训练集才有targets，测试集没有标签
        if self.has_label:
            self.targets = [
                self.class_to_idx[label]
                for label in self.dataframe["label"]
            ]
        else:
            self.targets = []

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        # CSV中记录的是images/0.jpg这样的相对路径
        relative_path = self.image_paths[index]

        # 拼接出完整路径
        image_path = os.path.join(
            self.data_dir,
            relative_path
        )

        # 读取图片，并统一转换成RGB三通道
        with Image.open(image_path) as image:
            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        # 训练集返回图片和数字标签
        if self.has_label:
            label = self.targets[index]
            return image, label

        # 测试集没有标签，只返回图片
        return image
###
###
###
# 统计训练集全部图片的RGB均值和标准差

data_dir = "/kaggle/input/competitions/classify-leaves"
train_csv_path = os.path.join(data_dir, "train.csv")

train_df = pd.read_csv(train_csv_path)

# 建立标签映射，用于构造LeavesDataset
classes = sorted(
    train_df["label"].unique().tolist()
)

class_to_idx = {
    label: idx
    for idx, label in enumerate(classes)
}

# 统计时不能使用数据增强，也不能提前Normalize
statistics_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

statistics_dataset = LeavesDataset(
    dataframe=train_df,
    data_dir=data_dir,
    transform=statistics_transform,
    class_to_idx=class_to_idx
)

statistics_iter = data.DataLoader(
    statistics_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

# RGB三个通道的像素总和
channel_sum = torch.zeros(
    3,
    dtype=torch.float64
)

# RGB三个通道的像素平方和
channel_square_sum = torch.zeros(
    3,
    dtype=torch.float64
)

# 每个通道的像素数量
pixel_count = 0

for x, y in statistics_iter:
    # x的形状：(batch_size, 3, 224, 224)
    x = x.to(torch.float64)

    channel_sum += x.sum(
        dim=(0, 2, 3)
    )

    channel_square_sum += (x ** 2).sum(
        dim=(0, 2, 3)
    )

    pixel_count += (
        x.size(0) *
        x.size(2) *
        x.size(3)
    )

# E[X]
mean_tensor = channel_sum / pixel_count

# Var(X) = E[X²] - E[X]²
variance_tensor = (
    channel_square_sum / pixel_count
    - mean_tensor ** 2
)

std_tensor = torch.sqrt(
    variance_tensor.clamp(min=0)
)

# 转换成列表，方便传给transforms.Normalize
mean = mean_tensor.tolist()
std = std_tensor.tolist()

print("训练集RGB均值：")
print(mean)

print("训练集RGB标准差：")
print(std)

###
###
###


def data_maker(
    data_dir,
    mean,
    std,
    image_size=224
):
    train_csv_path = os.path.join(
        data_dir,
        "train.csv"
    )

    test_csv_path = os.path.join(
        data_dir,
        "test.csv"
    )

    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    if not {"image", "label"}.issubset(
        train_df.columns
    ):
        raise ValueError(
            "train.csv必须包含image和label两列"
        )

    if "image" not in test_df.columns:
        raise ValueError(
            "test.csv必须包含image列"
        )

    classes = sorted(
        train_df["label"].unique().tolist()
    )

    class_to_idx = {
        label: idx
        for idx, label in enumerate(classes)
    }

    # 训练集：数据增强 + 标准化
    train_transform = transforms.Compose([
        transforms.Resize(
            (image_size, image_size)
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomVerticalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=15,
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=(255, 255, 255)
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    # 验证集和测试集：不使用随机增强
    valid_transform = transforms.Compose([
        transforms.Resize(
            (image_size, image_size)
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    # 使用train.csv，带训练增强
    train_dataset = LeavesDataset(
        dataframe=train_df,
        data_dir=data_dir,
        transform=train_transform,
        class_to_idx=class_to_idx
    )

    # 仍然使用train.csv，但不带随机增强
    valid_dataset = LeavesDataset(
        dataframe=train_df,
        data_dir=data_dir,
        transform=valid_transform,
        class_to_idx=class_to_idx
    )

    # 使用test.csv，没有标签
    test_dataset = LeavesDataset(
        dataframe=test_df,
        data_dir=data_dir,
        transform=valid_transform,
        class_to_idx=class_to_idx
    )

    return (
        train_dataset,
        valid_dataset,
        test_dataset
    )

训练集RGB均值：
[0.7578773939744068, 0.7780095841876465, 0.7590052397539914]
训练集RGB标准差：
[0.24793641911058625, 0.23481707818847483, 0.2628070116477634]


In [3]:
data_dir = "/kaggle/input/competitions/classify-leaves"

train_dataset, valid_dataset,test_dataset = data_maker(
    data_dir=data_dir,
    mean=mean,
    std=std,
    image_size=224
)

In [4]:
def try_gpu(i=0):
    if torch.cuda.device_count()>=i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device=try_gpu()

In [ ]:
def get_k_fold_data(
    k,
    i,
    train_dataset,
    valid_dataset=None,
    seed=42
):
    if k < 2:
        raise ValueError("k必须大于等于2")

    if not 0 <= i < k:
        raise ValueError(
            f"i必须在0到{k - 1}之间"
        )

    if valid_dataset is None:
        valid_dataset = train_dataset

    if len(train_dataset) != len(valid_dataset):
        raise ValueError(
            "train_dataset和valid_dataset必须对应相同样本"
        )

    if not hasattr(train_dataset, "targets"):
        raise ValueError(
            "train_dataset必须包含targets属性"
        )

    # 每张图片对应的数字标签
    labels = np.array(
        train_dataset.targets
    )

    # 创建分层K折划分器
    k_fold = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=seed
    )

    # X在这里只用于提供样本数量
    # 真正用于分层的是labels
    fold_indices = list(
        k_fold.split(
            np.zeros(len(labels)),
            labels
        )
    )

    # 取出第i折的训练索引和验证索引
    train_indices, valid_indices = fold_indices[i]

    # 训练索引从带数据增强的Dataset中取
    train_data = Subset(
        train_dataset,
        train_indices.tolist()
    )

    # 验证索引从无数据增强的Dataset中取
    valid_data = Subset(
        valid_dataset,
        valid_indices.tolist()
    )

    return train_data, valid_data

In [6]:
class residual_normal(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
    def forward(self,x):
        y=self.net(x)
        y=y+x
        return F.relu(y)
    
class residual(nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(in_channel,out_channel,kernel_size=3,stride=2,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(out_channel,out_channel,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(out_channel))
        self.net1 = nn.Sequential(
    nn.Conv2d(
        in_channel,
        out_channel,
        kernel_size=1,
        stride=2,
        bias=False
    ),
    nn.BatchNorm2d(out_channel)
)
    def forward(self,x):
        y=self.net(x)+self.net1(x)
        return F.relu(y)

In [7]:
def net_maker():
    b1 = nn.Sequential(nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
    b2=nn.Sequential(residual_normal(64,64),residual_normal(64,64))
    b3=nn.Sequential(residual(64,128),residual_normal(128,128))
    b4=nn.Sequential(residual(128,256),residual_normal(256,256))
    b5=nn.Sequential(residual(256,512),residual_normal(512,512))
    net=nn.Sequential(b1,b2,b3,b4,b5,nn.AdaptiveAvgPool2d((1,1)),nn.Flatten(),nn.Linear(512,176))
    return net
    
    

In [8]:
net = net_maker().to(device)

x = torch.randn(
    size=(2, 3, 224, 224),
    device=device
)

with torch.no_grad():
    y = net(x)

print(y.shape)

torch.Size([2, 176])


In [ ]:
def init(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(
            m.weight,
            mode="fan_out",
            nonlinearity="relu"  #表示后面接relu
        )

    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)

In [10]:
acc_list=[]
for i in range(k):
    train_data,valid_data=get_k_fold_data(k,i,train_dataset,valid_dataset=valid_dataset,seed=42)
    train_iter=data.DataLoader(train_data,shuffle=True,**loader_config)
    valid_iter=data.DataLoader(valid_data,shuffle=False,**loader_config)
    # train_eval_data = Subset(
    #     valid_dataset,
    #     train_data.indices
    # )

    # train_eval_iter = data.DataLoader(
    #     train_eval_data,
    #     batch_size=batch_size,
    #     shuffle=False,
    #     num_workers=4,
    #     pin_memory=False
    # )
    net=net_maker().to(device)
    net.apply(init)
    loss=nn.CrossEntropyLoss()
    trainer=torch.optim.SGD(net.parameters(),lr=lr,weight_decay=weight_decay,momentum=momentum)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    trainer,
    T_max=epoch,
    eta_min=1e-4
)
    for j in range(epoch):
        net.train()
        train_loss=0.0
        train_count=0
        train_acc=0.0
        for x,y in train_iter:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            trainer.zero_grad()
            logit=net(x)
            l=loss(logit,y)
            train_loss+=l.item()*len(y)
            train_count+=len(y)
            train_acc+=(logit.argmax(dim=1)==y).sum().item()
            l.backward()
            trainer.step()
        scheduler.step()
        print(f"第{i+1}折的第{j+1}个epoch的训练loss：{train_loss/train_count},准确率:{train_acc/train_count}")
        net.eval()
        with torch.no_grad():
            valid_loss=0.0
            valid_count=0
            valid_acc=0.0
            for x,y in valid_iter:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logit=net(x)
                l=loss(logit,y)
                valid_loss+=l.item()*len(y)
                valid_count+=len(y)
                valid_acc+=(logit.argmax(dim=1)==y).sum().item()
            print(f"第{i+1}折的第{j+1}个epoch的验证loss：{valid_loss/valid_count},准确率:{valid_acc/valid_count}")
        # update_bn(
        #     train_eval_iter,
        #     net,
        #     device=device
        # )
        # net.eval()
        # with torch.no_grad():
        #     valid_loss=0.0
        #     valid_count=0
        #     valid_acc=0.0
        #     for x,y in valid_iter:
        #         x = x.to(device, non_blocking=True)
        #         y = y.to(device, non_blocking=True)
        #         logit=net(x)
        #         l=loss(logit,y)
        #         valid_loss+=l.item()*len(y)
        #         valid_count+=len(y)
        #         valid_acc+=(logit.argmax(dim=1)==y).sum().item()
        #     print(f"第{i+1}折的第{j+1}个epoch的验证new_loss：{valid_loss/valid_count},准确率:{valid_acc/valid_count}")
            if len(acc_list) < j+1:
                acc_list.append(valid_acc/valid_count)
            else:
                acc_list[j]+=valid_acc/valid_count
        
    

第1折的第1个epoch的训练loss：4.446653636497952,准确率:0.06824683285655905
第1折的第1个epoch的验证loss：4.773884372924206,准确率:0.06810133478616182
第1折的第2个epoch的训练loss：3.212680568033823,准确率:0.19506879171774963
第1折的第2个epoch的验证loss：3.014725443583618,准确率:0.24353037319531462
第1折的第3个epoch的训练loss：2.5276438903133056,准确率:0.3146710257458112
第1折的第3个epoch的验证loss：3.7448723098621874,准确率:0.2002179242713157
第1折的第4个epoch的训练loss：2.0573921423162336,准确率:0.4201743631657812
第1折的第4个epoch的验证loss：3.0141698161317163,准确率:0.28166712067556526
第1折的第5个epoch的训练loss：1.7154427635102973,准确率:0.4985015665440676
第1折的第5个epoch的验证loss：2.1936021235293763,准确率:0.406973576682103
第1折的第6个epoch的训练loss：1.4885793984415356,准确率:0.5543522680833673
第1折的第6个epoch的验证loss：1.842250293329281,准确率:0.467992372650504
第1折的第7个epoch的训练loss：1.285352064825634,准确率:0.6092494210598011
第1折的第7个epoch的验证loss：2.389688244440094,准确率:0.4017978752383547
第1折的第8个epoch的训练loss：1.1361207558501076,准确率:0.6536575398447078
第1折的第8个epoch的验证loss：1.4184424585881723,准确率:0.5592481612639608
第1折的第9个epoch

In [11]:
max_acc=0.0
best_epoch=0
for i,acc in enumerate(acc_list):
    acc/=k
    if acc>max_acc:
        max_acc=acc
        best_epoch=i
net=net_maker().to(device)
net.apply(init)
loss=nn.CrossEntropyLoss()
trainer=torch.optim.SGD(net.parameters(),lr=lr,weight_decay=weight_decay,momentum=momentum)
final_iter=data.DataLoader(train_dataset,shuffle=True,**loader_config)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    trainer,
    T_max=epoch,
    eta_min=1e-4
)
for i in range(best_epoch+1):
    net.train()
    train_loss=0.0
    train_count=0
    train_acc=0.0
    for x,y in final_iter:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        trainer.zero_grad()
        logit=net(x)
        l=loss(logit,y)
        train_loss+=l.item()*len(y)
        train_count+=len(y)
        train_acc+=(logit.argmax(dim=1)==y).sum().item()
        l.backward()
        trainer.step()
    scheduler.step()
    print(f"第{i+1}个epoch的训练loss：{train_loss/train_count},准确率:{train_acc/train_count}")


第1个epoch的训练loss：4.130491982427129,准确率:0.10041954993733994
第2个epoch的训练loss：2.8333551499432184,准确率:0.2617010842913965
第3个epoch的训练loss：2.2014293609338127,准确率:0.3835340271345284
第4个epoch的训练loss：1.7807704515498204,准确率:0.48253691494578543
第5个epoch的训练loss：1.4938195835424888,准确率:0.5534789952596306
第6个epoch的训练loss：1.2500329334417728,准确率:0.6197351931564322
第7个epoch的训练loss：1.0959149045735843,准确率:0.6603280117691931
第8个epoch的训练loss：0.9702211351334377,准确率:0.696343921974609
第9个epoch的训练loss：0.857047264987634,准确率:0.7286547158502698
第10个epoch的训练loss：0.7730988226832989,准确率:0.756933471367079
第11个epoch的训练loss：0.6897300474878709,准确率:0.7834686427287092
第12个epoch的训练loss：0.6300703793535153,准确率:0.8039557565520623
第13个epoch的训练loss：0.5531120956746238,准确率:0.8271672206178826
第14个epoch的训练loss：0.49227427535535645,准确率:0.8483626655042772
第15个epoch的训练loss：0.46695170806765407,准确率:0.8552280281152945
第16个epoch的训练loss：0.41910474619014265,准确率:0.8711382335313028
第17个epoch的训练loss：0.38528846202413436,准确率:0.8847054977387893
第18个

In [12]:
save_path = "/kaggle/working/resnet18_scratch_v8.pth"

torch.save(
    {
        "model_state_dict": net.state_dict(),
        "best_epoch": best_epoch,
        "max_valid_acc": float(max_acc),
        "class_to_idx": train_dataset.class_to_idx,
        "idx_to_class": train_dataset.idx_to_class,
        "mean": mean,
        "std": std,
        "image_size": image_size
    },
    save_path
)

print("模型已保存到：", save_path)

模型已保存到： /kaggle/working/resnet18_scratch_v8.pth


In [13]:
# ============================================================
# 加载模型并预测测试集，生成Kaggle提交文件
# ============================================================

save_path = (
    "/kaggle/working/"
    "resnet18_scratch_v8.pth"
)

submission_path = (
    "/kaggle/working/"
    "submission.csv"
)


# 1. 加载checkpoint
checkpoint = torch.load(
    save_path,
    map_location=device
)

print(
    "模型最佳epoch：",
    checkpoint["best_epoch"]
)

print(
    "交叉验证最高平均准确率：",
    checkpoint["max_valid_acc"]
)


# 2. 重新创建模型并加载参数
predict_net = net_maker().to(device)

predict_net.load_state_dict(
    checkpoint["model_state_dict"]
)

# 预测时必须切换到eval模式
predict_net.eval()


# 3. 创建测试集DataLoader
test_iter = data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)


# 4. 预测测试集
predicted_indices = []

with torch.no_grad():
    for x in test_iter:
        x = x.to(
            device,
            non_blocking=True
        )

        logit = predict_net(x)

        prediction = logit.argmax(
            dim=1
        )

        # 暂时保存GPU上的预测结果
        predicted_indices.append(
            prediction
        )


# 合并所有batch，然后一次性传回CPU
predicted_indices = torch.cat(
    predicted_indices,
    dim=0
).cpu().tolist()


# 5. 将数字类别转换回字符串类别
idx_to_class = checkpoint[
    "idx_to_class"
]

predicted_labels = [
    idx_to_class[int(index)]
    for index in predicted_indices
]


# 6. 检查预测数量
assert (
    len(predicted_labels)
    == len(test_dataset)
), (
    "预测数量与测试集数量不一致："
    f"{len(predicted_labels)} != "
    f"{len(test_dataset)}"
)

print(
    "测试图片数量：",
    len(test_dataset)
)

print(
    "预测结果数量：",
    len(predicted_labels)
)


# 7. 建立图片路径到预测标签的映射
prediction_map = {
    image_path: label
    for image_path, label in zip(
        test_dataset.image_paths,
        predicted_labels
    )
}


# 8. 读取官方提交示例
sample_submission_path = os.path.join(
    data_dir,
    "sample_submission.csv"
)

submission = pd.read_csv(
    sample_submission_path
)


# 检查官方提交文件格式
if not {"image", "label"}.issubset(
    submission.columns
):
    raise ValueError(
        "sample_submission.csv"
        "必须包含image和label两列"
    )


# 按照官方提交文件的图片顺序填写预测结果
submission["label"] = (
    submission["image"]
    .map(prediction_map)
)


# 检查是否有图片没有匹配到预测结果
if submission["label"].isna().any():
    missing_images = submission.loc[
        submission["label"].isna(),
        "image"
    ].tolist()

    raise ValueError(
        "以下测试图片没有找到预测结果："
        f"{missing_images[:5]}"
    )


# 9. 保存提交文件
submission.to_csv(
    submission_path,
    index=False
)


# 10. 最终检查
print("\n提交文件前5行：")
print(submission.head())

print("\n提交文件大小：")
print(submission.shape)

print(
    "\n提交文件已保存到：",
    submission_path
)

模型最佳epoch： 38
交叉验证最高平均准确率： 0.9359776642466878
测试图片数量： 8800
预测结果数量： 8800

提交文件前5行：
              image                label
0  images/18353.jpg      asimina_triloba
1  images/18354.jpg         betula_nigra
2  images/18355.jpg  platanus_acerifolia
3  images/18356.jpg       pinus_bungeana
4  images/18357.jpg  platanus_acerifolia

提交文件大小：
(8800, 2)

提交文件已保存到： /kaggle/working/submission.csv
